# Swin detection-only fine-tuning at 512×768
Warm-start from the verified Swin detector checkpoint, or a DINO bundle containing that detector. Train detection only for 50 epochs. Random frame batches (not P-K) include all annotated people. No ReID training or crop encoding. Preserve the initial frame split seed; change no annotation labels.

Validation selects mean(AP50, AP75), each 101-point interpolated AP at a 0.001 candidate floor. This is not COCO AP50:95. Precision/recall use confidence 0.5. Final test is detection-only. For future person-search evaluation, load the saved scene resolution too; the older retrieval default of 256×384 must not be used silently.


In [6]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("github_token")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py", "offline_mining.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


Project imports configured from /kaggle/working/ML4CV-Assignment/src


In [10]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saadqasim2015 (models-alma-mater-studio) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [11]:
from pathlib import Path
import torch
import wandb

ENTITY = "models-alma-mater-studio"
PROJECT = "person-search"
SWIN_RUN_ID = "z2ewiyio"

training_run = wandb.Api().run(
    f"{ENTITY}/{PROJECT}/{SWIN_RUN_ID}"
)

matches = [
    a for a in training_run.logged_artifacts()
    if a.type == "model"
    and a.name.rsplit("/", 1)[-1].split(":")[0] == "prw-best-model"
]

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one best-model artifact, found {len(matches)}. "
        "Check the Swin run's output artifacts or saved Kaggle output."
    )

artifact = matches[0]
artifact_dir = Path(artifact.download(
    root=f"/kaggle/working/detector_checkpoints/{SWIN_RUN_ID}"
))
DETECTOR_CHECKPOINT = artifact_dir / "best.pt"


KeyboardInterrupt: 

In [ ]:
import random
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from model import PersonSearchModel
from prw_dataset import PRWPersonSearchDataset, PRWSubset, prw_collate_fn
from training import make_grad_scaler, start_run
from detection_training import detection_epoch, evaluate_detector

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SOURCE_CHECKPOINT=DETECTOR_CHECKPOINT
IMAGE_SIZE=(512,768)
BATCH_SIZE=16
NUM_EPOCHS=50
NUM_WORKERS=2
BACKBONE_LR=1e-5
HEAD_LR=1e-4
UPLOAD_EVERY=10
state=torch.load(SOURCE_CHECKPOINT,map_location='cpu',weights_only=False)
config=dict(state.get('detector_config',state.get('model_config',{})))
if config.get('backbone_name')!='swin_t':
    raise ValueError('Expected a verified Swin-T detector checkpoint')
SEED=int(state['run_config']['seed'])
VAL_FRACTION=float(state['run_config'].get('val_fraction',.10))
random.seed(SEED);torch.manual_seed(SEED)
config.update(pretrained=False,detection_only=True)
model=PersonSearchModel(**config).to(device)
model.load_state_dict(state.get('detector_state_dict',state.get('model_state_dict')),strict=True)
del state
full=PRWPersonSearchDataset(PRW_ROOT,split='train',image_size=IMAGE_SIZE,include_query_crops=False)
indices=torch.randperm(len(full),generator=torch.Generator().manual_seed(SEED)).tolist()
cut=round(len(indices)*VAL_FRACTION)
train_data=PRWSubset(full,indices[cut:]);val_data=PRWSubset(full,indices[:cut])
def loader(data,shuffle=False):
    return DataLoader(data,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,
                      collate_fn=prw_collate_fn,pin_memory=device.type=='cuda')
train_loader=loader(train_data,True);val_loader=loader(val_data)
print('Train/validation frames:',len(train_data),len(val_data),'seed:',SEED)
run_config=dict(experiment='swin_detection_only_512x768',seed=SEED,val_fraction=VAL_FRACTION,
                image_size=IMAGE_SIZE,batch_size=BATCH_SIZE,epochs=NUM_EPOCHS,
                source_checkpoint=str(SOURCE_CHECKPOINT),model_config=config,
                backbone_lr=BACKBONE_LR,head_lr=HEAD_LR,
                checkpoint_selection='mean_AP50_AP75')
run=start_run(run_config,name='swin-detector-only-512x768',mode='online')
# Baseline at the new resolution before any detector updates.
baseline=evaluate_detector(model,val_loader,device)
run.log({'epoch':0,**{f'val/{k}':v for k,v in baseline.items()}})
print('Initial validation:',baseline)


In [ ]:
optimizer=torch.optim.AdamW([
    {'params':[p for p in model.backbone.parameters() if p.requires_grad],'lr':BACKBONE_LR},
    {'params':[p for name,p in model.named_parameters()
               if p.requires_grad and not name.startswith('backbone.')],'lr':HEAD_LR},
],weight_decay=1e-4)
scaler=make_grad_scaler(device)
checkpoint_dir=Path(repo_path)/'checkpoints_swin_detection_512x768'
checkpoint_dir.mkdir(parents=True,exist_ok=True)
best=baseline['mean_AP50_AP75']
def save_checkpoint(epoch,metrics,path):
    torch.save(dict(epoch=epoch,model_config=config,model_state_dict=model.state_dict(),
                    optimizer_state_dict=optimizer.state_dict(),scaler_state_dict=scaler.state_dict(),
                    run_config=run_config,detector_image_size=IMAGE_SIZE,
                    best_detection_ap=best,validation_metrics=metrics),path)
# Preserve warm-start weights if fine-tuning never improves validation.
save_checkpoint(0,baseline,checkpoint_dir/'best.pt')
for epoch in range(1,NUM_EPOCHS+1):
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats(device)
    losses=detection_epoch(model,train_loader,optimizer,scaler,device)
    payload={'epoch':epoch,**{f'train/{k}':v for k,v in losses.items()}}
    if device.type=='cuda':
        payload['train/peak_allocated_gib']=torch.cuda.max_memory_allocated(device)/2**30
        payload['train/peak_reserved_gib']=torch.cuda.max_memory_reserved(device)/2**30
    metrics=evaluate_detector(model,val_loader,device)
    payload.update({f'val/{k}':v for k,v in metrics.items()})
    run.log(payload)
    if metrics['mean_AP50_AP75']>best:
        best=metrics['mean_AP50_AP75'];save_checkpoint(epoch,metrics,checkpoint_dir/'best.pt')
    save_checkpoint(epoch,metrics,checkpoint_dir/'latest.pt')
    print(f'Epoch {epoch}: {metrics}; best={best:.4f}',flush=True)
    if epoch%UPLOAD_EVERY==0 or epoch==NUM_EPOCHS:
        for file,collection in [('best.pt','swin-detector-512x768-best'),('latest.pt','swin-detector-512x768-latest')]:
            artifact=wandb.Artifact(collection,type='model',metadata={'upload_epoch':epoch})
            artifact.add_file(str(checkpoint_dir/file));run.log_artifact(artifact)


## Final detection test
Load the validation-selected detector and evaluate official test frames once. This does not evaluate ReID. The checkpoint contains the detector input resolution for later controlled Swin/DINO retrieval comparisons.


In [ ]:
best_state=torch.load(checkpoint_dir/'best.pt',map_location='cpu',weights_only=False)
model.load_state_dict(best_state['model_state_dict'],strict=True)
selected_epoch=best_state['epoch']
del best_state
test_data=PRWPersonSearchDataset(PRW_ROOT,split='test',image_size=IMAGE_SIZE,include_query_crops=False)
test_metrics=evaluate_detector(model,loader(test_data),device)
output=checkpoint_dir/'test_detection.json'
output.write_text(json.dumps(dict(checkpoint_epoch=selected_epoch,metrics=test_metrics),indent=2))
try:
    payload={f'test_detection/{k}':v for k,v in test_metrics.items()}
    payload['test_detection/checkpoint_epoch']=selected_epoch
    run.summary.update(payload);run.log(payload)
    table=wandb.Table(columns=['metric','value'],data=[[k,v] for k,v in test_metrics.items()])
    run.log({'test_detection/results':table,
             'test_detection/chart':wandb.plot.bar(table,'metric','value',title='Detection test metrics')})
    print('Selected epoch:',selected_epoch,'FINAL DETECTION TEST:',test_metrics)
finally:
    run.finish()


## Evaluate fine-tuned Swin detection + saved DINO ReID (no training)
For evaluation only, run the repository setup and W&B login/PRW_ROOT cells (1–2), then this section. Skip the fine-tuning and detection-test cells above. Those cells have been left unchanged.

Edit the two exact artifact versions and their source run IDs below, or supply local checkpoint paths. Swin weights come ONLY from the new detector checkpoint; DINO weights and pooling configuration come ONLY from the ReID checkpoint. The old detector inside the DINO bundle is deliberately not loaded. `best.pt` is a file; `v23` is an artifact version, not a filename.

Detector resolution is read from its checkpoint and passed explicitly to evaluate_prw. Query/gallery person-crop resolution comes from DINO's saved crop_size. Logging creates a separate evaluation run so neither training history is overwritten.


In [ ]:
from pathlib import Path
import gc
import importlib
import torch
import wandb
import prw_evaluation
importlib.reload(prw_evaluation)
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table, plot_retrieval_query
from model import PersonSearchModel
from dino_reid import DinoPersonSearch
from dinov3_reid import DinoV3CropReID

ENTITY = 'models-alma-mater-studio'
PROJECT = 'person-search'
SWIN_RUN_ID = '77d9acgi'
DINO_RUN_ID = 'xhdye71i'
SWIN_ARTIFACT = 'swin-detector-512x768-best:v0'
DINO_ARTIFACT = 'prw-dinov3-last4-p16k4-upper-lower-infonce-256x128-valmap-best-model:v23'
# Set paths to bypass artifact download, otherwise leave None.
SWIN_LOCAL_PATH = None
DINO_LOCAL_PATH = None
SCORE_THRESHOLD = .5
RUN_GT_COMPARISON = True
LOG_TO_WANDB = True
PREVIEW_QUERIES = 10

api = wandb.Api() if SWIN_LOCAL_PATH is None or DINO_LOCAL_PATH is None else None
def resolve_checkpoint(local_path, run_id, artifact_name):
    if local_path is not None:
        path = Path(local_path)
        source = {'path': str(path), 'artifact': None, 'run_id': None}
    else:
        source_run = api.run(f'{ENTITY}/{PROJECT}/{run_id}')
        matches = [a for a in source_run.logged_artifacts()
                   if a.type == 'model' and a.name.rsplit('/',1)[-1] == artifact_name]
        if len(matches) != 1:
            raise RuntimeError(f'{artifact_name} not uniquely logged by {run_id}; verify both values.')
        folder = Path('/kaggle/working/evaluation_checkpoints') / run_id / artifact_name.replace(':','-')
        path = Path(matches[0].download(root=str(folder))) / 'best.pt'
        source = {'path': str(path), 'artifact': artifact_name, 'run_id': run_id}
    if not path.is_file():
        raise FileNotFoundError(path)
    return path, source

swin_path, swin_source = resolve_checkpoint(SWIN_LOCAL_PATH, SWIN_RUN_ID, SWIN_ARTIFACT)
dino_path, dino_source = resolve_checkpoint(DINO_LOCAL_PATH, DINO_RUN_ID, DINO_ARTIFACT)
# Release previous models/optimizer if this section follows training in the same kernel.
for name in ('model','detector','reid','optimizer','best_state','state'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
swin_state = torch.load(swin_path, map_location='cpu', weights_only=False)
dino_state = torch.load(dino_path, map_location='cpu', weights_only=False)
detector_config = dict(swin_state['model_config'])
if detector_config.get('backbone_name') != 'swin_t':
    raise ValueError('Expected a Swin-T detector checkpoint')
scene_size = swin_state.get('detector_image_size', swin_state.get('run_config',{}).get('image_size'))
if scene_size is None:
    raise ValueError('Detector checkpoint has no saved scene size; verify its training configuration.')
DETECTOR_IMAGE_SIZE = tuple(int(v) for v in scene_size)
detector_config.update(pretrained=False, detection_only=True)
detector = PersonSearchModel(**detector_config)
detector.load_state_dict(swin_state['model_state_dict'], strict=True)
reid_config = dict(dino_state['reid_config'])
reid_config['pretrained'] = False
reid = DinoV3CropReID(**reid_config)
reid.load_state_dict(dino_state['reid_state_dict'], strict=True)
model = DinoPersonSearch(detector, reid).to(device).eval()
model.requires_grad_(False)
provenance = dict(detector=swin_source, reid=dino_source,
                  detector_epoch=swin_state.get('epoch'), reid_epoch=dino_state.get('epoch'),
                  detector_image_size=DETECTOR_IMAGE_SIZE,
                  reid_crop_size=reid.crop_size, feature_pooling=reid.feature_pooling,
                  metric_loss=dino_state.get('run_config',{}).get('metric_loss'),
                  score_threshold=SCORE_THRESHOLD, iou_threshold=.5)
del swin_state, dino_state
gc.collect()
print('Loaded separate detector/ReID checkpoints:', provenance)


In [ ]:
# Full official test queries/gallery. No optimization is performed.
import json
import pandas as pd
from IPython.display import display
comparison_rows = []
reid_diagnostic_records = {}
for mode in (('detected', 'gt') if RUN_GT_COMPARISON else ('detected',)):
    result = evaluate_prw(
        model, PRW_ROOT, device, gallery_mode=mode,
        detector_image_size=DETECTOR_IMAGE_SIZE,
        score_threshold=SCORE_THRESHOLD, iou_threshold=.5,
        batch_size=2, query_batch_size=64, ranking_batch_size=8,
        gallery_crop_batch_size=32, return_records=True, table_queries=100, table_top_k=5,
    )
    reid_diagnostic_records[mode] = result.pop('retrieval_records')
    comparison_rows.append(result)
    print(f"{mode}: mAP={result['retrieval_mAP']:.4f}, top-1={result['retrieval_top1']:.4f}")
display(pd.DataFrame(comparison_rows))
from datetime import datetime, timezone
output_dir = Path('/kaggle/working/evaluation_results') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'results.json').write_text(json.dumps(
    dict(provenance=provenance, results=comparison_rows, records=reid_diagnostic_records), indent=2))
print('Saved metrics, provenance and retrieval records:', output_dir)


In [ ]:
# Visualization and optional logging reuse saved results; no inference repeats.
import matplotlib.pyplot as plt
for mode, records in reid_diagnostic_records.items():
    for index in range(min(PREVIEW_QUERIES, len(records))):
        fig = plot_retrieval_query(records, PRW_ROOT, query_index=index, top_k=5)
        fig.suptitle(f'{mode} gallery')
        fig.savefig(output_dir / f'{mode}-{index:03d}.png', dpi=100, bbox_inches='tight')
        if index == 0: plt.show()
        plt.close(fig)
if LOG_TO_WANDB:
    if wandb.run is not None: wandb.finish()
    evaluation_run = wandb.init(entity=ENTITY, project=PROJECT,
                               name='finetuned-swin512-dino-retrieval', job_type='evaluation',
                               config=provenance)
    try:
        scalars = {f'{"test" if row["gallery_mode"] == "detected" else "test_gt"}/{key}': value
                   for row in comparison_rows for key,value in row.items()
                   if isinstance(value,(int,float))}
        # Familiar top-level final score names, in addition to detailed metrics.
        detected = next(row for row in comparison_rows if row['gallery_mode']=='detected')
        scalars.update({'test/mAP':detected['retrieval_mAP'], 'test/top1':detected['retrieval_top1']})
        evaluation_run.summary.update(scalars)
        evaluation_run.log(scalars)
        scores = wandb.Table(columns=['mode_metric','percent'],data=[
            [f"{row['gallery_mode']} {label}",100*row[key]] for row in comparison_rows
            for label,key in [('mAP','retrieval_mAP'),('Top1','retrieval_top1')]])
        evaluation_run.log({'test/final_scores':scores,
            'test/final_scores_chart':wandb.plot.bar(scores,'mode_metric','percent',title='Retrieval scores (%)')})
        for mode,records in reid_diagnostic_records.items():
            evaluation_run.log({f'{mode}/retrievals':make_wandb_retrieval_table(records,PRW_ROOT),
                                f'{mode}/images':[wandb.Image(str(p)) for p in sorted(output_dir.glob(f'{mode}-*.png'))]})
    finally:
        evaluation_run.finish()
